In [ ]:
from __future__ import annotations
 
import sys
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import pandas as pd
BASE_DIR = Path("../data/raw")

datasets = {
    "videos": [],
    "comments": [],
    "channels": [],
}

for csv_file in BASE_DIR.rglob("*.csv"):
    try:
        df = pd.read_csv(csv_file)

        df["source_file"] = csv_file.name
        df["source_path"] = str(csv_file)

        id_dir = next(
            (part for part in csv_file.parts if part.startswith("id_")),
            None
        )
        df["collection_id"] = id_dir

        stem_parts = csv_file.stem.split("_")
        country = stem_parts[-1] if len(stem_parts) > 2 else None
        df["country"] = country

        filename = csv_file.name.lower()

        if "video" in filename:
            datasets["videos"].append(df)

        elif "comment" in filename:
            datasets["comments"].append(df)

        elif "channel" in filename:
            datasets["channels"].append(df)

    except Exception as e:
        print(f"Erro ao ler {csv_file}: {e}")

videos_df = pd.concat(datasets["videos"], ignore_index=True) \
    if datasets["videos"] else pd.DataFrame()

comments_df = pd.concat(datasets["comments"], ignore_index=True) \
    if datasets["comments"] else pd.DataFrame()

channels_df = pd.concat(datasets["channels"], ignore_index=True) \
    if datasets["channels"] else pd.DataFrame()

print(f"Videos: {len(videos_df)}")
print(f"Comments: {len(comments_df)}")
print(f"Channels: {len(channels_df)}")


In [ ]:
from IPython.display import display, Markdown
display(Markdown("## visão geral e estrutura do dataset"))
videos_df.info() 

display(Markdown("---"))

display(Markdown("## amostra dos dados (primeiras 5 linhas)"))
display(videos_df.head())

display(Markdown("---"))

display(Markdown("##  rsumo estatístico (variáveis nméricas)"))
display(videos_df.describe()) 

if videos_df.select_dtypes(include=['object']).shape[1] > 0:
    display(Markdown("## resumo estatístico (variáveis categóricas)"))
    display(videos_df.describe())

In [ ]:
null_pct = df.isnull().mean().sort_values(ascending=False)
null_abs = df.isnull().sum().sort_values(ascending=False)
null_df = pd.DataFrame({"nulos": null_abs, "pct_%": (null_pct*100).round(1)})
display(null_df.to_string())


In [ ]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from datetime import datetime

In [ ]:
# display(Markdown("## LIMPEZA DE DADOS\n"))
 
# df_clean = videos_df.copy()

# df_clean = df_clean.replace(["", "None", "nan", "NaN", "unknown"], np.nan)

# cols_100_null = ["thumbnail", "channel_title"]
# df_clean.drop(columns=cols_100_null, inplace=True)
# display(Markdown(f"Removidas {len(cols_100_null)} colunas 100% nulas: {cols_100_null}"))

# df_clean = df_clean[df_clean['country'] != 'channel']

# dup_count = df_clean.duplicated(subset="video_id").sum()
# df_clean.drop_duplicates(subset="video_id", inplace=True)
# display(Markdown(f"Duplicatas removidas por video_id: {dup_count}"))
 
# df_clean["published_at"] = pd.to_datetime(df_clean["published_at"], errors="coerce")
# display(Markdown(f"published_at convertido para datetime"))
 
# LIKE_WEIGHT = 2
# COMMENT_WEIGHT = 5
# num_cols = ["view_count", "like_count", "comment_count", "official_category_id", "category_id"]
# for col in num_cols:
#     if col in df_clean.columns:
#         df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")
        
# df_clean["engagement_rate"] = np.where(
#     df_clean["view_count"] > 0,
#     (
#         df_clean["like_count"].fillna(0) * LIKE_WEIGHT +
#         df_clean["comment_count"].fillna(0) * COMMENT_WEIGHT
#     ) / df_clean["view_count"],
#     np.nan
# )
# df_clean["like_ratio"]    = df_clean["like_count"]    / df_clean["view_count"]
# df_clean["comment_ratio"] = df_clean["comment_count"] / df_clean["view_count"]
# display(Markdown(f"Criadas colunas: engagement_rate, like_ratio, comment_ratio"))
 
# df_clean["year"]  = df_clean["published_at"].dt.year
# df_clean["month"] = df_clean["published_at"].dt.month
# display(Markdown(f"Criadas colunas: year, month"))
 
# cap_views = df_clean["view_count"].quantile(0.995)
# outliers_n = (df_clean["view_count"] > cap_views).sum()
# df_clean["view_count_capped"] = df_clean["view_count"].clip(upper=cap_views)
# display(Markdown(f"view_count: {outliers_n} outliers capados no p99.5 ({cap_views:,.0f})"))
 
# # df_clean["source_label"] = df_clean["source_file"].str.replace(".csv","",regex=False)
# # display(Markdown(f"source_label criado"))
# display(Markdown(f"\nshape final: {df_clean.shape[0]} linhas x {df_clean.shape[1]} colunas"))

# POS_BASE_DIR = Path("../data/processed")
# POS_BASE_DIR.mkdir(parents=True, exist_ok=True)

# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# file_name = f"processed_videos_{timestamp}.csv"
# caminho_final = POS_BASE_DIR / file_name

# df_clean.to_csv(caminho_final, index=False, sep=',', encoding='utf-8')

# display(Markdown(f"\nsalvo no caminho: {caminho_final}"))



In [ ]:
df_clean = videos_df.copy()

df_clean = df_clean.replace(["", "None", "nan", "NaN", "unknown"], np.nan)

df_clean = df_clean[df_clean["country"] != "channel"]
df_clean = df_clean.drop_duplicates(subset="video_id")

df_clean = df_clean.drop(columns=["thumbnail", "channel_title"], errors="ignore")

df_clean["published_at"] = pd.to_datetime(df_clean["published_at"], errors="coerce")

num_cols = ["view_count", "like_count", "comment_count", "official_category_id", "category_id"]
for col in num_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

LIKE_WEIGHT = 2
COMMENT_WEIGHT = 5

valid_views = df_clean["view_count"].gt(0)

df_clean["engagement_rate"] = np.where(
    valid_views,
    (
        df_clean["like_count"].fillna(0) * LIKE_WEIGHT +
        df_clean["comment_count"].fillna(0) * COMMENT_WEIGHT
    ) / df_clean["view_count"],
    np.nan
)

df_clean["like_ratio"] = np.where(
    valid_views,
    df_clean["like_count"].fillna(0) / df_clean["view_count"],
    np.nan
)

df_clean["comment_ratio"] = np.where(
    valid_views,
    df_clean["comment_count"].fillna(0) / df_clean["view_count"],
    np.nan
)

df_clean["year"] = df_clean["published_at"].dt.year
df_clean["month"] = df_clean["published_at"].dt.month
df_clean["source_label"] = df_clean["source_file"].str.replace(".csv","",regex=False)

# cap_views = df_clean["view_count"].quantile(0.995)
cap_views = df_clean["view_count"].quantile(0.99)
df_clean["view_count_capped"] = df_clean["view_count"].clip(upper=cap_views)


imputer = SimpleImputer(strategy="median")

df_clean[["view_count", "like_count", "comment_count"]] = (
    imputer.fit_transform(
        df_clean[["view_count", "like_count", "comment_count"]]
    )
)

In [ ]:
display(Markdown(f"\nshape final: {df_clean.shape[0]} linhas x {df_clean.shape[1]} colunas"))

POS_BASE_DIR = Path("../data/processed")
POS_BASE_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = f"processed_videos_{timestamp}.csv"
caminho_final = POS_BASE_DIR / file_name

df_clean.to_csv(caminho_final, index=False, sep=',', encoding='utf-8')

display(Markdown(f"\nsalvo no caminho: {caminho_final}"))

In [ ]:
display(Markdown("## visão geral e estrutura do dataset"))
df_clean.info() 

display(Markdown("---"))

display(Markdown("## amostra dos dados (primeiras 5 linhas)"))
display(df_clean.head())

display(Markdown("---"))

display(Markdown("##  rsumo estatístico (variáveis nméricas)"))
display(df_clean.describe()) 

if df_clean.select_dtypes(include=['object']).shape[1] > 0:
    display(Markdown("## resumo estatístico (variáveis categóricas)"))
    display(df_clean.describe())

display(Markdown("##  colunas e etc"))
display(df_clean.columns)

In [ ]:

ROOT: Path = Path.cwd().parent
sys.path.append(str(ROOT))

from src.core.plot_eda import YouTubeDashboard

In [ ]:
dashboard = YouTubeDashboard(df_clean)
figures = dashboard.generate_separate_plots(output_dir="meus_plots")
# plt.show()
# plt.close(fig)


In [ ]:
# df_clean.loc[df_clean['country'] == 'channel', 'country']
        

In [ ]:
display(figures[1])
display(figures[2])
display(figures[4])
display(figures[6])
display(figures[7])
display(figures[8])